# 01 — Data Loading & Quality Assessment

**Project:** Flight Arrival Delay Prediction  
**Data Source:** BTS On-Time Performance (January–June 2025)  
**Target Variable:** `ArrDel15` — binary indicator, 1 if arrival delay ≥ 15 minutes  

---

This notebook covers:
1. Loading and concatenating six monthly BTS CSV files
2. Standardising column names
3. Quality checks (shapes, dtypes, value ranges, missingness, duplicates, monthly counts)
4. A documented cleaning pipeline with row-count checkpoints
5. Building a leakage-safe model dataset and saving it as Parquet

All decisions about which columns to keep or drop are logged in `agent_logs/decision_register.md`.

## 0. Imports & Configuration

In [34]:
import glob
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_DIR       = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
OUTPUT_PATH   = os.path.join(PROCESSED_DIR, 'model_dataset.parquet')

os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

Libraries loaded.
NumPy  : 2.1.3
Pandas : 2.2.3


## 1. Load Raw CSV Files

BTS publishes one CSV per month. We discover all CSVs in `data/raw/` automatically using `glob`, sort them so they are read in chronological order, and concatenate them into a single DataFrame.

> **Why glob?** Hard-coding six filenames is brittle; glob picks up the files regardless of exact naming conventions and makes it trivial to add more months later.

In [35]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))

print(f'Found {len(csv_files)} CSV file(s):')
for f in csv_files:
    print(f'  {os.path.basename(f)}')

Found 7 CSV file(s):
  T_ONTIME_REPORTING_APRIL.csv
  T_ONTIME_REPORTING_FEV.csv
  T_ONTIME_REPORTING_JAN.csv
  T_ONTIME_REPORTING_JULY.csv
  T_ONTIME_REPORTING_JUNE.csv
  T_ONTIME_REPORTING_MAY.csv
  T_ONTIME_REPORTING_march.csv


In [36]:
# Read each file, store in a list, then concatenate once — cheaper than repeated pd.concat inside a loop
frames = []
for path in csv_files:
    month_df = pd.read_csv(
        path,
        low_memory=False,   # avoids mixed-type warnings on large files
    )
    frames.append(month_df)
    print(f'  {os.path.basename(path):45s}  rows={len(month_df):>8,}  cols={month_df.shape[1]}')

raw = pd.concat(frames, ignore_index=True)
print(f'\nConcatenated shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns')

  T_ONTIME_REPORTING_APRIL.csv                   rows= 583,950  cols=54
  T_ONTIME_REPORTING_FEV.csv                     rows= 504,884  cols=65
  T_ONTIME_REPORTING_JAN.csv                     rows= 539,747  cols=54
  T_ONTIME_REPORTING_JULY.csv                    rows= 631,428  cols=54
  T_ONTIME_REPORTING_JUNE.csv                    rows= 611,575  cols=54
  T_ONTIME_REPORTING_MAY.csv                     rows= 605,648  cols=60
  T_ONTIME_REPORTING_march.csv                   rows= 600,872  cols=54

Concatenated shape: 4,078,104 rows × 72 columns


## 2. Column Renaming

BTS uses `UPPER_WITH_UNDERSCORES` naming. We rename to a mixed-case convention that is more readable in code and consistent with common BTS documentation references (e.g., `ArrDel15`, `DepDelay`).

Only the columns listed in the project specification are renamed; any extra BTS columns are left as-is and will naturally be excluded when we select the final feature set.

In [37]:
RENAME_MAP = {
    'YEAR'                : 'Year',
    'QUARTER'             : 'Quarter',
    'MONTH'               : 'Month',
    'DAY_OF_MONTH'        : 'DayofMonth',
    'DAY_OF_WEEK'         : 'DayOfWeek',
    'FL_DATE'             : 'FlightDate',
    'OP_UNIQUE_CARRIER'   : 'Reporting_Airline',
    'TAIL_NUM'            : 'Tail_Number',
    'ORIGIN_AIRPORT_ID'   : 'OriginAirportID',
    'ORIGIN'              : 'Origin',
    'ORIGIN_STATE_ABR'    : 'OriginState',
    'DEST_AIRPORT_ID'     : 'DestAirportID',
    'DEST'                : 'Dest',
    'DEST_STATE_ABR'      : 'DestState',
    'CRS_DEP_TIME'        : 'CRSDepTime',
    'DEP_DELAY'           : 'DepDelay',
    'DEP_DEL15'           : 'DepDel15',
    'DEP_TIME_BLK'        : 'DepTimeBlk',
    'CRS_ARR_TIME'        : 'CRSArrTime',
    'ARR_DELAY'           : 'ArrDelay',
    'ARR_DEL15'           : 'ArrDel15',
    'CRS_ELAPSED_TIME'    : 'CRSElapsedTime',
    'DISTANCE'            : 'Distance',
    'DISTANCE_GROUP'      : 'DistanceGroup',
    'CANCELLED'           : 'Cancelled',
    'DIVERTED'            : 'Diverted',
    'CARRIER_DELAY'       : 'CarrierDelay',
    'WEATHER_DELAY'       : 'WeatherDelay',
    'NAS_DELAY'           : 'NASDelay',
    'SECURITY_DELAY'      : 'SecurityDelay',
    'LATE_AIRCRAFT_DELAY' : 'LateAircraftDelay',
}

# Only rename columns that actually exist to avoid KeyErrors on optional BTS fields
rename_existing = {k: v for k, v in RENAME_MAP.items() if k in raw.columns}
raw = raw.rename(columns=rename_existing)

print(f'Renamed {len(rename_existing)} columns.')
missing_from_map = set(RENAME_MAP.keys()) - set(rename_existing.keys())
if missing_from_map:
    print(f'Columns in rename map but NOT in data: {missing_from_map}')

Renamed 31 columns.


## 3. Quality Checks

Before any cleaning we freeze a snapshot of the raw data and run four diagnostic checks:

- **3.1** Shape and dtypes
- **3.2** Value-range validation for key categorical/flag columns
- **3.3** Missingness per column
- **3.4** Duplicate detection on the flight uniqueness key
- **3.5** Month-by-month row counts

### 3.1 Shape & Data Types

In [38]:
print(f'Shape : {raw.shape[0]:,} rows × {raw.shape[1]} columns')
print()
print('Column dtypes:')
print(raw.dtypes.to_string())

Shape : 4,078,104 rows × 72 columns

Column dtypes:
Year                         int64
Quarter                      int64
Month                        int64
DayofMonth                   int64
DayOfWeek                    int64
FlightDate                  object
Reporting_Airline           object
Tail_Number                 object
OP_CARRIER_FL_NUM          float64
OriginAirportID              int64
ORIGIN_AIRPORT_SEQ_ID        int64
ORIGIN_CITY_MARKET_ID        int64
Origin                      object
ORIGIN_CITY_NAME            object
OriginState                 object
ORIGIN_STATE_NM             object
DestAirportID                int64
DEST_AIRPORT_SEQ_ID          int64
DEST_CITY_MARKET_ID          int64
Dest                        object
DEST_CITY_NAME              object
DestState                   object
DEST_STATE_NM               object
CRSDepTime                   int64
DEP_TIME                   float64
DepDelay                   float64
DEP_DELAY_NEW              float64
Dep

In [39]:
print('First 5 rows:')
raw.head(5)

First 5 rows:


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,OP_CARRIER_FL_NUM,OriginAirportID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,Origin,ORIGIN_CITY_NAME,OriginState,ORIGIN_STATE_NM,DestAirportID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,Dest,DEST_CITY_NAME,DestState,DEST_STATE_NM,CRSDepTime,DEP_TIME,...,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,OP_CARRIER_AIRLINE_ID,OP_CARRIER,DIV_AIRPORT_LANDINGS,DIV_REACHED_DEST,DIV_ACTUAL_ELAPSED_TIME,DIV_ARR_DELAY,DIV_DISTANCE,DIV1_AIRPORT,DIV1_AIRPORT_ID,DIV1_AIRPORT_SEQ_ID,DIV1_WHEELS_ON,DIV1_WHEELS_OFF,ORIGIN_STATE_FIPS,DEST_STATE_FIPS,ACTUAL_ELAPSED_TIME,FIRST_DEP_TIME,TOTAL_ADD_GTIME,LONGEST_ADD_GTIME
0,2025,2,4,1,2,4/1/2025 12:00:00 AM,AA,N101NN,12.00,12892,1289208,32575,LAX,"Los Angeles, CA",CA,California,10721,1072102,30721,BOS,"Boston, MA",MA,Massachusetts,835,828.00,...,2611.00,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,4,1,2,4/1/2025 12:00:00 AM,AA,N101NN,1578.00,10721,1072102,30721,BOS,"Boston, MA",MA,Massachusetts,12892,1289208,32575,LAX,"Los Angeles, CA",CA,California,1800,1753.00,...,2611.00,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,2,4,1,2,4/1/2025 12:00:00 AM,AA,N101NN,28.00,12892,1289208,32575,LAX,"Los Angeles, CA",CA,California,12478,1247805,31703,JFK,"New York, NY",NY,New York,2259,2255.00,...,2475.00,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,2,4,1,2,4/1/2025 12:00:00 AM,AA,N102NN,16.00,14771,1477104,32457,SFO,"San Francisco, CA",CA,California,12478,1247805,31703,JFK,"New York, NY",NY,New York,1037,1031.00,...,2586.00,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,2,4,1,2,4/1/2025 12:00:00 AM,AA,N102NN,177.00,12478,1247805,31703,JFK,"New York, NY",NY,New York,14771,1477104,32457,SFO,"San Francisco, CA",CA,California,2030,2020.00,...,2586.00,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Value-Range Validation

We check the logical bounds for the calendar and flag columns. Violations indicate either upstream data issues or unexpected file contents.

In [40]:
range_checks = [
    ('Month',      1,    12),
    ('DayofMonth', 1,    31),
    ('DayOfWeek',  1,     7),
    ('Cancelled',  0,     1),
    ('Diverted',   0,     1),
]

print(f'{'Column':<20} {'Min':>10} {'Max':>10} {'Out-of-range':>15}')
print('-' * 60)
for col, lo, hi in range_checks:
    if col not in raw.columns:
        print(f'{col:<20} {'MISSING COLUMN':>35}')
        continue
    col_clean = pd.to_numeric(raw[col], errors='coerce')
    col_min   = col_clean.min()
    col_max   = col_clean.max()
    out       = ((col_clean < lo) | (col_clean > hi)).sum()
    flag      = '  <<< CHECK' if out > 0 else ''
    print(f'{col:<20} {col_min:>10.0f} {col_max:>10.0f} {out:>15,}{flag}')

Column                      Min        Max    Out-of-range
------------------------------------------------------------
Month                         1          7               0
DayofMonth                    1         31               0
DayOfWeek                     1          7               0
Cancelled                     0          1               0
Diverted                      0          1               0


### 3.3 Missingness Per Column

A sorted table of null counts helps prioritise which columns need attention. Columns with 100% missingness after filtering (e.g., delay-cause columns for non-delayed flights) are expected.

In [41]:
null_counts = raw.isnull().sum()
null_pct    = (null_counts / len(raw) * 100).round(2)

missingness = (
    pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
    .sort_values('null_count', ascending=False)
)

# Show only columns with at least one null
print('Columns with missing values (sorted by count):')
display(missingness[missingness['null_count'] > 0])

Columns with missing values (sorted by count):


,null_count,null_pct
DIV_ARR_DELAY,4077237,99.98
DIV_ACTUAL_ELAPSED_TIME,4077237,99.98
DIV1_WHEELS_OFF,4077223,99.98
DIV_REACHED_DEST,4077101,99.98
DIV_DISTANCE,4077101,99.98
DIV1_AIRPORT_SEQ_ID,4077011,99.97
DIV1_AIRPORT,4077011,99.97
DIV1_AIRPORT_ID,4077011,99.97
DIV1_WHEELS_ON,4077011,99.97
TOTAL_ADD_GTIME,4073374,99.88


### 3.4 Duplicate Detection

A flight is conceptually unique if (FlightDate, Reporting_Airline, Origin, Dest, CRSDepTime) is unique — two rows sharing all five values represent the same scheduled flight.

In [42]:
DUP_KEY = ['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime']

# Check which key columns are present
missing_key_cols = [c for c in DUP_KEY if c not in raw.columns]
if missing_key_cols:
    print(f'WARNING: key columns not found in data: {missing_key_cols}')
else:
    n_dups = raw.duplicated(subset=DUP_KEY).sum()
    print(f'Duplicate rows on key {DUP_KEY}:')
    print(f'  {n_dups:,} duplicate rows ({n_dups / len(raw) * 100:.3f}% of total)')
    if n_dups > 0:
        print('\nSample of duplicated records:')
        dup_mask = raw.duplicated(subset=DUP_KEY, keep=False)
        display(raw[dup_mask][DUP_KEY + ['ArrDel15']].sort_values(DUP_KEY).head(10))

Duplicate rows on key ['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime']:
  1,462 duplicate rows (0.036% of total)

Sample of duplicated records:


,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,ArrDel15
1097934,1/1/2025 12:00:00 AM,OO,PSC,SEA,500,0.00
1098300,1/1/2025 12:00:00 AM,OO,PSC,SEA,500,0.00
1097804,1/1/2025 12:00:00 AM,OO,SAN,LAX,615,1.00
1098136,1/1/2025 12:00:00 AM,OO,SAN,LAX,615,0.00
1261453,1/10/2025 12:00:00 AM,OO,SEA,LAX,600,1.00
1262843,1/10/2025 12:00:00 AM,OO,SEA,LAX,600,NaN
1269321,1/10/2025 12:00:00 AM,YX,CMH,LGA,600,0.00
1269685,1/10/2025 12:00:00 AM,YX,CMH,LGA,600,0.00
1268796,1/10/2025 12:00:00 AM,YX,DCA,BOS,659,0.00
1269667,1/10/2025 12:00:00 AM,YX,DCA,BOS,659,0.00


### 3.5 Month-by-Month Row Counts

This sanity check confirms that each month contributed a reasonable number of rows and that no month was accidentally loaded twice or missed.

In [43]:
monthly = (
    raw.groupby('Month', sort=True)
    .size()
    .rename('row_count')
    .reset_index()
)
monthly['month_name'] = monthly['Month'].map({
    1: 'January', 2: 'February', 3: 'March',
    4: 'April',   5: 'May',      6: 'June',
    7: 'July',    8: 'August',   9: 'September',
    10: 'October', 11: 'November', 12: 'December'
})
monthly['cumulative'] = monthly['row_count'].cumsum()

print('Month-by-month row counts:')
display(monthly[['Month', 'month_name', 'row_count', 'cumulative']])
print(f'\nTotal: {monthly["row_count"].sum():,}')

Month-by-month row counts:


,Month,month_name,row_count,cumulative
0,1,January,539747,539747
1,2,February,504884,1044631
2,3,March,600872,1645503
3,4,April,583950,2229453
4,5,May,605648,2835101
5,6,June,611575,3446676
6,7,July,631428,4078104



Total: 4,078,104


## 4. Cleaning Pipeline

We clean the data in six documented steps. After each step we print the row count removed so you can trace exactly where data was lost.

| Step | Action | Rationale |
|------|--------|-----------|
| 1 | Remove cancelled flights | No arrival outcome |
| 2 | Remove diverted flights | Arrival airport differs from plan |
| 3 | Drop rows where `ArrDel15` is NaN | Target is unobserved |
| 4 | Parse `FlightDate` to datetime | Required for feature engineering later |
| 5 | Filter invalid Distance / time values | Likely data-entry errors |
| 6 | Deduplicate on flight key | Keep first occurrence |

In [44]:
df = raw.copy()
cleaning_log = []

def checkpoint(df, step_name, rows_before):
    rows_after   = len(df)
    rows_removed = rows_before - rows_after
    cleaning_log.append({
        'step'         : step_name,
        'rows_before'  : rows_before,
        'rows_removed' : rows_removed,
        'rows_after'   : rows_after,
        'pct_removed'  : rows_removed / rows_before * 100 if rows_before > 0 else 0,
    })
    return rows_after

rows = len(df)
print(f'Starting rows: {rows:,}')

Starting rows: 4,078,104


In [45]:
# Step 1: Remove cancelled flights (Cancelled == 1)
rows_before = rows
df = df[df['Cancelled'] != 1].copy()
rows = checkpoint(df, 'Step 1: Remove cancelled flights', rows_before)
print(f'Step 1 complete — rows remaining: {rows:,}')

Step 1 complete — rows remaining: 4,010,997


In [46]:
# Step 2: Remove diverted flights (Diverted == 1)
rows_before = rows
df = df[df['Diverted'] != 1].copy()
rows = checkpoint(df, 'Step 2: Remove diverted flights', rows_before)
print(f'Step 2 complete — rows remaining: {rows:,}')

Step 2 complete — rows remaining: 3,998,633


In [47]:
# Step 3: Drop rows where ArrDel15 is NaN (target unobserved)
rows_before = rows
df = df.dropna(subset=['ArrDel15']).copy()
rows = checkpoint(df, 'Step 3: Drop missing ArrDel15', rows_before)
print(f'Step 3 complete — rows remaining: {rows:,}')

Step 3 complete — rows remaining: 3,998,633


In [48]:
row_count = len(df)
print(row_count)

3998633


In [49]:
# Step 4: Parse FlightDate to datetime (auto-detect BTS format)
rows_before = rows
# BTS files encode FlightDate as 'M/D/YYYY 12:00:00 AM' (e.g., '1/9/2025 12:00:00 AM')
# Do not specify format, so pandas infers correctly. Normalize to strip time.
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce').dt.normalize()
n_bad_dates = df['FlightDate'].isna().sum()
if n_bad_dates > 0:
    print(f'  {n_bad_dates:,} rows had unparseable FlightDate — dropping them.')
    df = df.dropna(subset=['FlightDate'])
rows = checkpoint(df, 'Step 4: Parse FlightDate', rows_before)
print(f'Step 4 complete — rows remaining: {rows:,}')
print(f'  FlightDate range: {df["FlightDate"].min().date()} to {df["FlightDate"].max().date()}')

Step 4 complete — rows remaining: 3,998,633
  FlightDate range: 2025-01-01 to 2025-07-31


Here we had an issue where the agent incorrectly read the data - our data was in date time format but when trying to change a date format it assumed our normal date format was Year-month-day without time which allowed it to be unparseable and that consequently led to dropping the rows and delting the data.

In [50]:
# Step 5: Remove rows with invalid Distance, CRSDepTime, or CRSArrTime
rows_before = rows

distance_ok  = pd.to_numeric(df['Distance'],    errors='coerce').fillna(0) > 0
dep_time_ok  = pd.to_numeric(df['CRSDepTime'],  errors='coerce').between(0, 2359, inclusive='both')
arr_time_ok  = pd.to_numeric(df['CRSArrTime'],  errors='coerce').between(0, 2359, inclusive='both')

valid_mask = distance_ok & dep_time_ok & arr_time_ok
df = df[valid_mask].copy()
rows = checkpoint(df, 'Step 5: Remove invalid Distance/CRSDepTime/CRSArrTime', rows_before)
print(f'Step 5 complete — rows remaining: {rows:,}')

Step 5 complete — rows remaining: 3,998,632


In [51]:
# Step 6: Deduplicate on flight key — keep first occurrence
rows_before = rows
dup_key_present = [c for c in DUP_KEY if c in df.columns]
df = df.drop_duplicates(subset=dup_key_present, keep='first').copy()
rows = checkpoint(df, 'Step 6: Deduplicate on flight key', rows_before)
print(f'Step 6 complete — rows remaining: {rows:,}')

Step 6 complete — rows remaining: 3,997,256


### 4.1 Cleaning Summary

In [52]:
cleaning_summary = pd.DataFrame(cleaning_log)
cleaning_summary['pct_removed'] = cleaning_summary['pct_removed'].map('{:.3f}%'.format)

print('=== Cleaning Pipeline Summary ===')
display(cleaning_summary)

total_removed = len(raw) - len(df)
print(f'\nTotal rows removed : {total_removed:,}  ({total_removed / len(raw) * 100:.2f}% of raw)')
print(f'Final clean rows   : {len(df):,}')

=== Cleaning Pipeline Summary ===


,step,rows_before,rows_removed,rows_after,pct_removed
0,Step 1: Remove cancelled flights,4078104,67107,4010997,1.646%
1,Step 2: Remove diverted flights,4010997,12364,3998633,0.308%
2,Step 3: Drop missing ArrDel15,3998633,0,3998633,0.000%
3,Step 4: Parse FlightDate,3998633,0,3998633,0.000%
4,Step 5: Remove invalid Distance/CRSDepTime/CRS...,3998633,1,3998632,0.000%
5,Step 6: Deduplicate on flight key,3998632,1376,3997256,0.034%



Total rows removed : 80,848  (1.98% of raw)
Final clean rows   : 3,997,256


## 5. Build Leakage-Safe Model Dataset

A core discipline in predictive modelling is ensuring that no information available only **after** the event leaks into the training features. For flight delay prediction, the "event" is departure — anything that occurs once the plane leaves the gate is off-limits.

**Excluded columns (post-departure actuals):**

| Column | Reason |
|--------|--------|
| `DepDelay` | Actual departure delay — unknown at prediction time |
| `DepDel15` | Binary version of `DepDelay` — same issue |
| `ArrDelay` | Actual arrival delay (in minutes) — this *is* the target |
| `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` | BTS delay cause codes — assigned post-flight |
| `Cancelled`, `Diverted` | Already filtered out; including them would be redundant |

**Derived features added:**

- `CRSDepHour` — hour component of `CRSDepTime` (0–23)
- `CRSDepMinute` — minute component of `CRSDepTime` (0–59)

In [53]:
# Derive hour and minute from scheduled departure time
df['CRSDepTime']   = pd.to_numeric(df['CRSDepTime'], errors='coerce')
df['CRSDepHour']   = (df['CRSDepTime'] // 100).astype('Int16')
df['CRSDepMinute'] = (df['CRSDepTime'] %  100).astype('Int16')

print('Derived features added: CRSDepHour, CRSDepMinute')
print(df[['CRSDepTime', 'CRSDepHour', 'CRSDepMinute']].head(5))

Derived features added: CRSDepHour, CRSDepMinute
   CRSDepTime  CRSDepHour  CRSDepMinute
0         835           8            35
1        1800          18             0
2        2259          22            59
3        1037          10            37
4        2030          20            30


In [54]:
# Define the final leakage-safe feature set
MODEL_COLUMNS = [
    # Calendar
    'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
    # Carrier
    'Reporting_Airline', 'Tail_Number',
    # Route
    'Origin', 'Dest', 'OriginAirportID', 'DestAirportID', 'OriginState', 'DestState',
    # Schedule
    'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime', 'Distance', 'DistanceGroup', 'DepTimeBlk',
    # Derived
    'CRSDepHour', 'CRSDepMinute',
    # Target
    'ArrDel15',
]

# Keep only columns that exist in the cleaned DataFrame
final_cols = [c for c in MODEL_COLUMNS if c in df.columns]
missing_cols = [c for c in MODEL_COLUMNS if c not in df.columns]

if missing_cols:
    print(f'WARNING — expected columns not found, will be omitted: {missing_cols}')

# Defence in depth: assert no post-departure leakage columns in model dataset
LEAKAGE_COLS = ['DepDelay', 'ArrDelay', 'DepDel15']
leaked = [c for c in LEAKAGE_COLS if c in final_cols]
assert len(leaked) == 0, f'LEAKAGE: post-departure columns in model dataset: {leaked}'

model_df = df[final_cols].copy()

# Cast ArrDel15 to int8 (binary 0/1 target)
model_df['ArrDel15'] = model_df['ArrDel15'].astype('int8')

print(f'Model dataset shape: {model_df.shape[0]:,} rows × {model_df.shape[1]} columns')
print(f'\nTarget distribution (ArrDel15):')
vc = model_df['ArrDel15'].value_counts().sort_index()
for val, count in vc.items():
    label = 'On-time (0)' if val == 0 else 'Delayed (1)'
    print(f'  {label}: {count:>9,}  ({count / len(model_df) * 100:.1f}%)')

Model dataset shape: 3,997,256 rows × 23 columns

Target distribution (ArrDel15):
  On-time (0): 3,079,161  (77.0%)
  Delayed (1):   918,095  (23.0%)


In [55]:
print('Final model dataset columns and dtypes:')
print(model_df.dtypes.to_string())

Final model dataset columns and dtypes:
Year                          int64
Quarter                       int64
Month                         int64
DayofMonth                    int64
DayOfWeek                     int64
FlightDate           datetime64[ns]
Reporting_Airline            object
Tail_Number                  object
Origin                       object
Dest                         object
OriginAirportID               int64
DestAirportID                 int64
OriginState                  object
DestState                    object
CRSDepTime                    int64
CRSArrTime                    int64
CRSElapsedTime              float64
Distance                    float64
DistanceGroup                 int64
DepTimeBlk                   object
CRSDepHour                    Int16
CRSDepMinute                  Int16
ArrDel15                       int8


## 6. Save to Parquet

Parquet is chosen over CSV for the processed dataset because:
- It preserves column dtypes (especially `datetime64` for `FlightDate` and `int8` for `ArrDel15`)
- Snappy compression reduces file size ~5× compared to raw CSV
- Column-oriented storage makes downstream column-subset reads fast

In [56]:
model_df.to_parquet(OUTPUT_PATH, index=False, compression='snappy')

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
print(f'Saved: {OUTPUT_PATH}')
print(f'File size: {file_size_mb:.1f} MB')

# Verification round-trip
verify = pd.read_parquet(OUTPUT_PATH)
assert verify.shape == model_df.shape, 'Shape mismatch on verification read!'
assert list(verify.columns) == list(model_df.columns), 'Column mismatch on verification read!'
print(f'Verification passed — {verify.shape[0]:,} rows × {verify.shape[1]} columns read back correctly.')

Saved: ../data/processed/model_dataset.parquet
File size: 59.4 MB
Verification passed — 3,997,256 rows × 23 columns read back correctly.


## 7. Summary

This notebook has:

1. **Loaded** 6 monthly BTS CSV files from `data/raw/` and concatenated them into a single DataFrame
2. **Renamed** 31 columns from BTS `UPPER_UNDERSCORE` convention to a readable mixed-case convention
3. **Checked** data quality across shape, dtypes, value ranges, missingness, duplicates, and monthly counts
4. **Cleaned** the data through 6 documented steps, removing cancelled/diverted flights, missing targets, invalid values, and duplicates
5. **Selected** a leakage-safe feature set (pre-departure only) with two derived time features
6. **Saved** the processed dataset to `data/processed/model_dataset.parquet`

**Next step:** `02_exploratory_data_analysis.ipynb` — univariate and bivariate analysis of the feature set.